# 98. 主成分分析（PCA）

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 13 / 34 步：扩展监督/无监督模型工具箱**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** K-Means聚类  →  **本章任务：** 主成分分析（PCA）  →  **下一步：** 交叉验证与超参数调优
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

做数据分析时常常会遇到一个麻烦：一张表有几十上百个特征，画图看不过来，计算也会变慢。



## 本章目标

学完本章，你将能够：

- **理解**：理解「主成分分析（PCA）」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「主成分分析（PCA）」的关键输出指标。
- **迁移**：能把「主成分分析（PCA）」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 98.1 核心概念

**背景引入**：做数据分析时常常会遇到一个麻烦：一张表有几十上百个特征，画图看不过来，计算也会变慢。主成分分析（PCA）就是专门用来“减负”的方法，它把重复且相关的列压缩成少数几个互不相关的新成分，让我们先在更低的维度上看清整体结构。学懂它之后，你就能判断每个新成分保留了多少信息，自然更容易评估降维到底值不值得。


- PCA 是无监督线性投影
- 主成分彼此正交且按方差排序（打个比方：把一堆又重复又相关的指标，拧成少数几个互不重复的“综合分”，按保留信息多少排队；取前几个，就能在平面上一眼看懂原本几十维的数据。）
- 载荷表示原特征与成分的线性组合
- 高解释方差不保证最适合预测目标


## 98.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| Wine 二维投影 | `pca.transform()`、`pca.explained_variance_ratio_.round()`、`pca.explained_variance_ratio_.sum()`、`pd.DataFrame()` | 标准化后查看前两个主成分及累计解释方差。 | 未标准化导致大尺度特征支配主成分 |
| 载荷与成分数 | `pd.DataFrame()`、`loadings.PC1.abs()`、`.sort_values()`、`.head()` | 查看每个主成分主要由哪些原始特征构成。 | 把主成分解释成因果因子 |


## 98.3 示例 1：Wine 二维投影

标准化后查看前两个主成分及累计解释方差。


<!-- math-foundation:chapter-98 -->
### 数学推导｜PCA 寻找方差最大的正交方向

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜先中心化数据并计算协方差。** $\Sigma=X_c^TX_c/(n-1)$。

**第 2 步｜寻找单位方向 $v$ 上方差最大者。** 投影 $z=X_cv$ 的方差为 $v^T\Sigma v$，所以求解

$$
\max_{\lVert v\rVert=1}v^T\Sigma v
$$

**第 3 步｜使用拉格朗日乘子。** 对 $v^T\Sigma v-\lambda(v^Tv-1)$ 求导，得到 $\Sigma v=\lambda v$。最大特征值对应第一主成分，后续方向再加与前面方向正交的约束。

**第 4 步｜用特征值计算解释方差比。** $EVR_k=\lambda_k/\sum_j\lambda_j$。

**把上面的关系收束为本章计算式：**

$$
\Sigma v_k=\lambda_kv_k,\qquad z_k=Xv_k,\qquad EVR_k=\frac{\lambda_k}{\sum_j\lambda_j}
$$

**符号解释：** $v_k$ 是第 $k$ 个主成分方向，$\lambda_k$ 是其解释方差。

**代码对应：** 标准化后拟合 `PCA`，用 `explained_variance_ratio_` 决定保留维度。

**使用边界：** PCA 是无监督线性投影；高方差方向不一定最有业务或预测价值。


In [ ]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

data = load_wine(as_frame=True)
X, y = data.data, data.target
Xs = StandardScaler().fit_transform(X)
pca = PCA(n_components=2).fit(Xs)
Z = pca.transform(Xs)
print(
    "解释方差:",
    pca.explained_variance_ratio_.round(3),
    "累计:",
    round(pca.explained_variance_ratio_.sum(), 3),
)
display(pd.DataFrame(Z, columns=["PC1", "PC2"]).assign(target=y).head())


**练一练**：把“标准化”这一步去掉，看看二维投影的解释方差怎么变。

示例里我们先对 Wine 的 13 个特征做 `StandardScaler` 标准化，再拟合成二维主成分。现在请反过来做一次：直接对原始数据 `X` 拟合 `PCA(n_components=2)`，把它跟标准化版本的累计解释方差（`explained_variance_ratio_.sum()`）放在一起比较。把你观察到的“变大 / 变小”以及原因记录下来，完成后可对照下方隐藏答案。


In [ ]:
# 请在下方填写代码

# 任务：比较“不标准化直接投影”与“标准化后投影”的累计解释方差，并解释差异。
import pandas as pd
from sklearn.decomposition import PCA

# 1) 不标准化：直接对原始数据 X 拟合二维 PCA，并求出累计解释方差
raw_pca = PCA(n_components=2).fit(X)  # 填写：用原始数据 X 拟合
raw_sum = raw_pca.explained_variance_ratio_.sum()  # 填写：累计解释方差

# 2) 标准化：示例里已经用变量 pca 算好了，直接取累计
std_sum = pca.explained_variance_ratio_.sum()

# 3) 观察与原因
observed_change = ""  # 填写：标准化让累计解释方差“更大”还是“更小”？
reason = ""  # 填写：用一句话说明为什么


In [ ]:
import pandas as pd
from sklearn.decomposition import PCA

# 1) 不标准化：直接对原始数据 X 拟合二维 PCA，并求出累计解释方差
raw_pca = PCA(n_components=2).fit(X)
raw_sum = raw_pca.explained_variance_ratio_.sum()

# 2) 标准化：复用示例中已经拟合好的变量 pca
std_sum = pca.explained_variance_ratio_.sum()

observed_change = "标准化后累计解释方差变小"
reason = "未标准化时，Wine 中量级大的特征（如 proline 等）会主导第一个主成分，"
reason += "前两个成分就吸收了绝大部分方差；标准化把每个特征拉到相同尺度后，"
reason += "方差被摊开到更多成分上，所以前两维的累计占比反而下降。"

print("不标准化累计: %.3f  标准化累计: %.3f" % (raw_sum, std_sum))
print("我的观察：", observed_change, "|", reason)


## 98.4 示例 2：载荷与成分数

查看每个主成分主要由哪些原始特征构成。


In [ ]:
loadings = pd.DataFrame(
    pca.components_.T, index=X.columns, columns=["PC1", "PC2"]
)
print("PC1绝对载荷最高:")
display(loadings.PC1.abs().sort_values(ascending=False).head())
pca95 = PCA(n_components=0.95).fit(Xs)
print("达到95%累计解释方差需要成分数:", pca95.n_components_)


## 98.5 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 98.6 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 98.7 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 98.7.1 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 98.7.2 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 98.8 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 98.8.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 98.9 易错点提醒

- 未标准化导致大尺度特征支配主成分
- 把主成分解释成因果因子
- 切分前拟合 PCA 造成泄漏
- 仅凭解释方差决定预测性能


## 98.10 练习与作业

1. 构建 StandardScaler + PCA(0.9) + LogisticRegression
2. 使用分层测试集
3. 报告保留成分数和准确率

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 98.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“构建 StandardScaler + PCA(0.9) + LogisticRegression”。
2. **独立完成**：不复制示例代码，完成“使用分层测试集”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“报告保留成分数和准确率”，用一两句话说明你修改了什么。

### 98.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 98.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
# 练习使用 Wine 三分类数据做分层切分（避免复用实训中仅 6 行、无法分层的 X/y）
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
X, y = wine.data, wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=87
)
pca_model = make_pipeline(
    StandardScaler(), PCA(n_components=0.9), LogisticRegression(max_iter=500)
).fit(X_train, y_train)
practice_score = pca_model.score(X_test, y_test)
kept = pca_model.named_steps["pca"].n_components_
print("成分数:", kept, "准确率:", round(practice_score, 3))


## 98.12 小结

使用 PCA 将相关特征压缩为少数正交主成分，理解解释方差、载荷和降维流水线。


### 98.12.1 你已经掌握

- 标准化后拟合 PCA
- 解释 explained_variance_ratio
- 选择累计解释方差阈值
- 将 PCA 放入预测流水线避免泄漏


### 98.12.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 98.12.3 需要注意

- 未标准化导致大尺度特征支配主成分
- 把主成分解释成因果因子
- 切分前拟合 PCA 造成泄漏
- 仅凭解释方差决定预测性能


### 98.12.4 完成检查

- [ ] 能够标准化后拟合 PCA
- [ ] 能够解释 explained_variance_ratio
- [ ] 能够选择累计解释方差阈值
- [ ] 能够将 PCA 放入预测流水线避免泄漏


### 98.12.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
